# Freight Rate Prediction

This notebook develops a machine learning pipeline to predict freight rates from shipment, route, equipment, weight, and date information.

The project will use historical freight data to train and evaluate models before generating rate predictions for December shipments.

## 1. Load and Inspect the Data

First, we load the four provided datasets and inspect their basic structure.

This helps us understand:
- How many rows and columns each dataset contains
- Which variables are available
- What the data looks like
- How the training, validation, and December prediction datasets differ

We will use this initial investigation to guide our feature engineering and modeling strategy.

In [3]:
import pandas as pd
import numpy as np
import os

# ============================================================
# 1. FILE PATHS
# ============================================================

# The data folder is located inside the GitHub repository.
# Using a relative path makes the notebook portable and
# independent of the user's computer-specific file path.

DATA_DIR = "./data"

TRAIN_PATH = os.path.join(
    DATA_DIR,
    "train-test.csv"
)

VALIDATION_PATH = os.path.join(
    DATA_DIR,
    "validation.csv"
)

DECEMBER_PATH = os.path.join(
    DATA_DIR,
    "december-chart-inputs.csv"
)

TEMPLATE_PATH = os.path.join(
    DATA_DIR,
    "validation-predictions-template.csv"
)


# ============================================================
# 2. VERIFY DATA FILES EXIST
# ============================================================

print("=" * 70)
print("CHECKING DATA FILES")
print("=" * 70)

required_files = {
    "Training data": TRAIN_PATH,
    "Validation data": VALIDATION_PATH,
    "December data": DECEMBER_PATH,
    "Validation template": TEMPLATE_PATH
}

for name, path in required_files.items():

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"{name} not found at: {path}\n"
            "Make sure the required file is inside the "
            "'data' folder."
        )

    print(f"{name}: OK")


# ============================================================
# 3. LOAD DATA
# ============================================================

train = pd.read_csv(
    TRAIN_PATH
)

validation = pd.read_csv(
    VALIDATION_PATH
)

december = pd.read_csv(
    DECEMBER_PATH
)

template = pd.read_csv(
    TEMPLATE_PATH
)


# ============================================================
# 4. BASIC INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET SHAPES")
print("=" * 70)

print(
    f"Training data:   {train.shape}"
)

print(
    f"Validation data: {validation.shape}"
)

print(
    f"December data:   {december.shape}"
)

print(
    f"Template:        {template.shape}"
)


# ============================================================
# 5. COLUMN INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("TRAINING COLUMNS")
print("=" * 70)

print(
    train.columns.tolist()
)


print("\n" + "=" * 70)
print("VALIDATION COLUMNS")
print("=" * 70)

print(
    validation.columns.tolist()
)


print("\n" + "=" * 70)
print("DECEMBER COLUMNS")
print("=" * 70)

print(
    december.columns.tolist()
)


print("\n" + "=" * 70)
print("TEMPLATE COLUMNS")
print("=" * 70)

print(
    template.columns.tolist()
)


# ============================================================
# 6. FIRST 5 ROWS
# ============================================================

print("\n" + "=" * 70)
print("TRAINING SAMPLE")
print("=" * 70)

display(
    train.head()
)


print("\n" + "=" * 70)
print("VALIDATION SAMPLE")
print("=" * 70)

display(
    validation.head()
)


print("\n" + "=" * 70)
print("DECEMBER SAMPLE")
print("=" * 70)

display(
    december.head()
)


print("\n" + "=" * 70)
print("TEMPLATE SAMPLE")
print("=" * 70)

display(
    template.head()
)

CHECKING DATA FILES
Training data: OK
Validation data: OK
December data: OK
Validation template: OK

DATASET SHAPES
Training data:   (48000, 14)
Validation data: (12000, 13)
December data:   (31, 7)
Template:        (12000, 2)

TRAINING COLUMNS
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']

VALIDATION COLUMNS
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']

DECEMBER COLUMNS
['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']

TEMPLATE COLUMNS
['load_id', 'predicted_rate']

TRAINING SAMPLE


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28



VALIDATION SAMPLE


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,2025-11-01,0.88117,1.95480
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,2025-11-01,0.89156,2.14579
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,2025-11-01,0.88308,2.22541



DECEMBER SAMPLE


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,NaN
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,NaN



TEMPLATE SAMPLE


,load_id,predicted_rate
0,TE-000001,NaN
1,TE-000002,NaN
2,TE-000003,NaN
3,TE-000004,NaN
4,TE-000005,NaN


## 2. Check Data Quality

Before building any models, we check the datasets for missing values, duplicate records, incorrect data types, and unusual values.

This helps ensure that the model is trained on reliable data and identifies any preprocessing steps we need before feature engineering.

In [4]:
# ============================================================
# 2. DATA QUALITY CHECK
# ============================================================

datasets = {
    "Training": train,
    "Validation": validation,
    "December": december,
    "Template": template
}

for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(f"{name.upper()} DATA QUALITY")
    print("=" * 70)

    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")

    print("\nMissing values:")
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No missing values")
    else:
        print(missing)

    print(f"\nDuplicate rows: {df.duplicated().sum():,}")

    print("\nData types:")
    print(df.dtypes)


TRAINING DATA QUALITY
Rows: 48,000
Columns: 14

Missing values:
weight          300
market_index    374
dtype: int64



Duplicate rows: 0

Data types:
load_id             str
pickup              str
delivery            str
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment           str
weight          float64
date                str
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object

VALIDATION DATA QUALITY
Rows: 12,000
Columns: 13

Missing values:
weight          165
market_index    249
dtype: int64

Duplicate rows: 0

Data types:
load_id             str
pickup              str
delivery            str
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment           str
weight          float64
date                str
market_index    float64
quote_signal    float64
dtype: object

DECEMBER DATA QUALITY
Rows: 31
Columns: 7

Missing values:
predicted_rate    31
dtype: int64

Duplicate rows: 0

Data types:
pickup        

## 3. Analyze the Target Variable

The target variable is `posted_rate`, which represents the historical freight rate.

We examine its summary statistics and distribution to understand the typical rate, range, variability, and potential outliers in the training data.

This information will help guide our modeling and evaluation strategy.

In [5]:
# ============================================================
# 3. TARGET VARIABLE ANALYSIS
# ============================================================

target = train["posted_rate"]

print("=" * 70)
print("POSTED RATE SUMMARY")
print("=" * 70)

print(target.describe())

print("\n" + "=" * 70)
print("TARGET QUANTILES")
print("=" * 70)

print(
    target.quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print("\n" + "=" * 70)
print("TARGET INFORMATION")
print("=" * 70)

print(f"Mean:   ${target.mean():,.2f}")
print(f"Median: ${target.median():,.2f}")
print(f"Min:    ${target.min():,.2f}")
print(f"Max:    ${target.max():,.2f}")
print(f"Std:    ${target.std():,.2f}")

POSTED RATE SUMMARY
count    48000.000000
mean      2373.980682
std       1486.493245
min         57.220000
25%       1251.555000
50%       2030.760000
75%       3330.750000
max      25533.000000
Name: posted_rate, dtype: float64

TARGET QUANTILES
0.01     327.1688
0.05     599.7385
0.25    1251.5550
0.50    2030.7600
0.75    3330.7500
0.95    4953.7665
0.99    5972.8340
Name: posted_rate, dtype: float64

TARGET INFORMATION
Mean:   $2,373.98
Median: $2,030.76
Min:    $57.22
Max:    $25,533.00
Std:    $1,486.49


## 4. Explore Key Features

Freight rates are likely influenced by shipment characteristics, route information, market conditions, and timing.

We examine the main numerical and categorical features to understand their ranges, distributions, and relationships with the target variable.

These observations will help us identify useful features for the modeling stage.

In [6]:
# ============================================================
# 4. KEY FEATURE ANALYSIS
# ============================================================

print("=" * 70)
print("NUMERICAL FEATURE SUMMARY")
print("=" * 70)

numeric_cols = [
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "posted_rate"
]

display(train[numeric_cols].describe().T)


print("\n" + "=" * 70)
print("EQUIPMENT TYPES")
print("=" * 70)

print(train["equipment"].value_counts())


print("\n" + "=" * 70)
print("UNIQUE LOCATIONS")
print("=" * 70)

print(f"Unique pickup locations:   {train['pickup'].nunique():,}")
print(f"Unique delivery locations: {train['delivery'].nunique():,}")


print("\n" + "=" * 70)
print("DATE RANGE")
print("=" * 70)

train_dates = pd.to_datetime(train["date"])

print(f"Earliest date: {train_dates.min()}")
print(f"Latest date:   {train_dates.max()}")


print("\n" + "=" * 70)
print("TOP PICKUP LOCATIONS")
print("=" * 70)

print(train["pickup"].value_counts().head(10))


print("\n" + "=" * 70)
print("TOP DELIVERY LOCATIONS")
print("=" * 70)

print(train["delivery"].value_counts().head(10))

NUMERICAL FEATURE SUMMARY


,count,mean,std,min,25%,50%,75%,max
distance,48000.0,1135.856654,728.564416,70.00000,550.40000,953.30000,1645.525000,3439.80000
weight,47700.0,31028.844004,9391.440620,-47500.00000,25800.00000,31436.50000,37018.000000,47500.00000
market_index,47626.0,1.083387,0.168091,0.67639,0.94967,1.05580,1.219590,1.46778
quote_signal,48000.0,2.062468,0.291391,0.69228,1.89103,2.05575,2.221685,3.61035
posted_rate,48000.0,2373.980682,1486.493245,57.22000,1251.55500,2030.76000,3330.750000,25533.00000



EQUIPMENT TYPES
equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64

UNIQUE LOCATIONS
Unique pickup locations:   64
Unique delivery locations: 64

DATE RANGE
Earliest date: 2025-01-01 00:00:00
Latest date:   2025-10-31 00:00:00

TOP PICKUP LOCATIONS
pickup
Oklahoma City    1242
Lexington        1209
Bakersfield      1193
Fort Wayne       1170
Hartford         1150
Richmond         1140
Nashville        1124
Phoenix          1121
Baton Rouge      1115
Mobile           1094
Name: count, dtype: int64

TOP DELIVERY LOCATIONS
delivery
Lexington        1197
Fort Wayne       1176
Baton Rouge      1167
Bakersfield      1156
Hartford         1143
Oklahoma City    1140
Richmond         1109
Atlanta          1096
Phoenix          1090
Mobile           1089
Name: count, dtype: int64


## 5. Investigate Weight Anomalies

The initial feature analysis revealed negative values in the `weight` column, which are unusual because shipment weight would normally be expected to be positive.

Rather than immediately removing these observations, we investigate their frequency and relationship with the target. This helps determine whether they represent data errors, intentional patterns, or values that should be handled through feature engineering.

In [7]:
# ============================================================
# 5. INVESTIGATE WEIGHT ANOMALIES
# ============================================================

print("=" * 70)
print("NEGATIVE WEIGHT ANALYSIS")
print("=" * 70)

negative_weight = train[train["weight"] < 0]

print(f"Negative-weight rows: {len(negative_weight):,}")
print(f"Percentage of training data: {len(negative_weight) / len(train) * 100:.2f}%")

print("\nNegative weight summary:")
display(negative_weight["weight"].describe())

print("\nSample negative-weight records:")
display(
    negative_weight[
        [
            "load_id",
            "pickup",
            "delivery",
            "distance",
            "equipment",
            "weight",
            "date",
            "market_index",
            "quote_signal",
            "posted_rate"
        ]
    ].head(10)
)

print("\n" + "=" * 70)
print("TARGET COMPARISON")
print("=" * 70)

comparison = train.assign(
    weight_group=np.where(train["weight"] < 0, "Negative", "Non-negative")
).groupby("weight_group")["posted_rate"].agg(
    ["count", "mean", "median", "min", "max"]
)

display(comparison)

NEGATIVE WEIGHT ANALYSIS
Negative-weight rows: 292
Percentage of training data: 0.61%

Negative weight summary:


count      292.000000
mean    -31724.195205
std       8262.536326
min     -47500.000000
25%     -37284.250000
50%     -31821.500000
75%     -25928.500000
max      -5000.000000
Name: weight, dtype: float64


Sample negative-weight records:


,load_id,pickup,delivery,distance,equipment,weight,date,market_index,quote_signal,posted_rate
68,TR-000069,Amarillo,Dayton,1334.2,Reefer,-36559.0,2025-01-01,0.96102,2.11777,2831.27
204,TR-000205,Fresno,Tucson,496.2,Dry Van,-26670.0,2025-01-02,0.96903,2.13155,1046.92
206,TR-000207,Columbia,Raleigh,411.3,Reefer,-20003.0,2025-01-02,0.94800,2.31682,959.15
225,TR-000226,Greensboro,Lubbock,1317.4,Reefer,-23981.0,2025-01-02,1.00820,2.13918,2836.00
376,TR-000377,Madison,Providence,1331.1,Dry Van,-41397.0,2025-01-03,0.94787,1.91867,2522.54
446,TR-000447,Albuquerque,Albany,2438.3,Reefer,-37215.0,2025-01-03,0.96575,1.96959,4841.57
495,TR-000496,Montgomery,Dayton,715.9,Dry Van,-31214.0,2025-01-04,0.85153,2.03947,1473.78
641,TR-000642,Tulsa,Indianapolis,582.4,Dry Van,-41023.0,2025-01-05,0.81988,2.22721,1264.33
806,TR-000807,Cincinnati,Buffalo,676.1,Dry Van,-28320.0,2025-01-06,0.79158,1.95680,1307.76
1093,TR-001094,Cincinnati,Albany,1058.5,Dry Van,-25153.0,2025-01-07,0.89891,1.81081,1907.83



TARGET COMPARISON


,count,mean,median,min,max
weight_group,,,,,
Negative,292,2389.786233,1994.96,247.37,14561.21
Non-negative,47708,2373.883943,2031.16,57.22,25533.00


## 6. Analyze Historical Routes

Freight pricing can vary substantially between different origin-destination pairs.

We investigate how frequently routes repeat in the historical data and whether validation and December routes have appeared previously in the training period.

This will help us determine whether historical route-level pricing information can be used as a predictive feature.

In [8]:
# ============================================================
# 6. ROUTE ANALYSIS
# ============================================================

# Create a route identifier
train["route"] = train["pickup"] + " → " + train["delivery"]
validation["route"] = validation["pickup"] + " → " + validation["delivery"]
december["route"] = december["pickup"] + " → " + december["delivery"]


print("=" * 70)
print("ROUTE STATISTICS")
print("=" * 70)

print(f"Unique training routes:   {train['route'].nunique():,}")
print(f"Unique validation routes: {validation['route'].nunique():,}")
print(f"Unique December routes:   {december['route'].nunique():,}")


# ============================================================
# ROUTE FREQUENCY
# ============================================================

print("\n" + "=" * 70)
print("MOST COMMON TRAINING ROUTES")
print("=" * 70)

route_counts = train["route"].value_counts()

display(route_counts.head(15))


# ============================================================
# ROUTE REPEAT RATE
# ============================================================

print("\n" + "=" * 70)
print("ROUTE REPETITION")
print("=" * 70)

single_routes = (route_counts == 1).sum()
repeated_routes = (route_counts > 1).sum()

print(f"Routes appearing once:     {single_routes:,}")
print(f"Routes appearing multiple times: {repeated_routes:,}")


# ============================================================
# VALIDATION ROUTE OVERLAP
# ============================================================

training_routes = set(train["route"])

validation["route_seen_in_training"] = validation["route"].isin(training_routes)
december["route_seen_in_training"] = december["route"].isin(training_routes)

print("\n" + "=" * 70)
print("VALIDATION ROUTE OVERLAP")
print("=" * 70)

print(
    validation["route_seen_in_training"]
    .value_counts()
    .rename({True: "Seen in training", False: "New route"})
)


# ============================================================
# DECEMBER ROUTE OVERLAP
# ============================================================

print("\n" + "=" * 70)
print("DECEMBER ROUTE OVERLAP")
print("=" * 70)

print(
    december["route_seen_in_training"]
    .value_counts()
    .rename({True: "Seen in training", False: "New route"})
)


# ============================================================
# DECEMBER ROUTES
# ============================================================

print("\n" + "=" * 70)
print("DECEMBER ROUTES")
print("=" * 70)

display(
    december[
        [
            "pickup",
            "delivery",
            "distance",
            "equipment",
            "weight",
            "date",
            "route_seen_in_training"
        ]
    ]
)

ROUTE STATISTICS
Unique training routes:   4,014
Unique validation routes: 4,214
Unique December routes:   1

MOST COMMON TRAINING ROUTES


route
Phoenix → Shreveport         39
Columbia → Oklahoma City     39
Lexington → Atlanta          39
Fort Wayne → Philadelphia    38
Richmond → Oklahoma City     37
Fort Wayne → Hartford        37
Richmond → Phoenix           37
Lexington → Cincinnati       37
Shreveport → Hartford        37
Lexington → Kansas City      37
Mobile → Phoenix             36
Fort Wayne → Cincinnati      36
Oklahoma City → Atlanta      35
Oklahoma City → Nashville    35
Columbia → Montgomery        35
Name: count, dtype: int64


ROUTE REPETITION
Routes appearing once:     67
Routes appearing multiple times: 3,947

VALIDATION ROUTE OVERLAP
route_seen_in_training
Seen in training    10539
New route            1461
Name: count, dtype: int64

DECEMBER ROUTE OVERLAP
route_seen_in_training
Seen in training    31
Name: count, dtype: int64

DECEMBER ROUTES


,pickup,delivery,distance,equipment,weight,date,route_seen_in_training
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,True
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,True
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,True
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,True
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,True
5,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-06,True
6,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-07,True
7,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-08,True
8,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-09,True
9,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-10,True


## 7. Analyze Pricing Patterns Over Time

The training data covers January through October, while validation represents November and the final prediction period is December.

We examine how freight rates change over time and across shipment characteristics to identify seasonal and pricing patterns that may help predict future rates.

In [9]:
# ============================================================
# 7. TIME AND PRICING PATTERNS
# ============================================================

# Work with a datetime copy
train["date"] = pd.to_datetime(train["date"])

# Create month information
train["month"] = train["date"].dt.month
train["month_name"] = train["date"].dt.strftime("%B")


# ============================================================
# AVERAGE RATE BY MONTH
# ============================================================

print("=" * 70)
print("AVERAGE RATE BY MONTH")
print("=" * 70)

monthly_rates = (
    train.groupby(["month", "month_name"])["posted_rate"]
    .agg(["count", "mean", "median"])
    .reset_index()
    .sort_values("month")
)

display(monthly_rates)


# ============================================================
# RATE BY EQUIPMENT
# ============================================================

print("\n" + "=" * 70)
print("RATE BY EQUIPMENT")
print("=" * 70)

equipment_rates = (
    train.groupby("equipment")["posted_rate"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(equipment_rates)


# ============================================================
# RATE BY DISTANCE RANGE
# ============================================================

print("\n" + "=" * 70)
print("RATE BY DISTANCE RANGE")
print("=" * 70)

train["distance_group"] = pd.cut(
    train["distance"],
    bins=[0, 500, 1000, 1500, 2000, 2500, 3000, np.inf],
    labels=[
        "0-500",
        "500-1000",
        "1000-1500",
        "1500-2000",
        "2000-2500",
        "2500-3000",
        "3000+"
    ]
)

distance_rates = (
    train.groupby("distance_group", observed=True)["posted_rate"]
    .agg(["count", "mean", "median"])
)

display(distance_rates)


# ============================================================
# RATE BY WEIGHT RANGE
# ============================================================

print("\n" + "=" * 70)
print("RATE BY WEIGHT RANGE")
print("=" * 70)

train["weight_group"] = pd.cut(
    train["weight"],
    bins=[-np.inf, 0, 10000, 20000, 30000, 40000, 50000, np.inf],
    labels=[
        "Negative",
        "0-10k",
        "10k-20k",
        "20k-30k",
        "30k-40k",
        "40k-50k",
        "50k+"
    ]
)

weight_rates = (
    train.groupby("weight_group", observed=True)["posted_rate"]
    .agg(["count", "mean", "median"])
)

display(weight_rates)

AVERAGE RATE BY MONTH


,month,month_name,count,mean,median
0,1,January,4918,2255.967048,1915.200
1,2,February,4337,2273.804801,1994.250
2,3,March,5036,2372.268092,2022.920
3,4,April,4819,2372.162308,2044.240
4,5,May,4913,2421.776342,2065.510
5,6,June,4783,2497.030115,2120.220
6,7,July,4912,2415.162030,2059.145
7,8,August,4759,2338.407661,2015.730
8,9,September,4670,2406.374013,2057.130
9,10,October,4853,2379.051374,2035.900



RATE BY EQUIPMENT


,count,mean,median,std
equipment,,,,
Reefer,12045,2553.636939,2196.670,1589.670824
Flatbed,8753,2445.087223,2076.810,1505.053087
Dry Van,27202,2271.548686,1953.035,1423.029914



RATE BY DISTANCE RANGE


,count,mean,median
distance_group,,,
0-500,10290,819.909492,834.675
500-1000,15055,1659.697505,1632.280
1000-1500,9008,2584.268010,2515.930
1500-2000,6100,3583.066025,3517.815
2000-2500,4862,4319.928926,4250.650
2500-3000,2116,5204.678214,5106.840
3000+,569,5931.591599,5820.130



RATE BY WEIGHT RANGE


,count,mean,median
weight_group,,,
Negative,292,2389.786233,1994.960
0-10k,202,2223.714604,1841.345
10k-20k,3548,2267.459267,1954.350
20k-30k,16584,2322.298346,1986.065
30k-40k,20003,2405.422961,2060.700
40k-50k,7071,2464.325889,2094.590


## 8. Analyze the December Target Route

The December prediction dataset contains the same Lexington-to-Fort Wayne shipment repeated across all 31 days of December.

Because this route appears in the historical training data, we can investigate its previous rates, equipment, weight, distance, and temporal behavior.

This provides a useful benchmark and may support the development of historical route-level features.

In [10]:
# ============================================================
# 8. DECEMBER ROUTE ANALYSIS
# ============================================================

target_pickup = "Lexington"
target_delivery = "Fort Wayne"

route_history = train[
    (train["pickup"] == target_pickup) &
    (train["delivery"] == target_delivery)
].copy()

route_history = route_history.sort_values("date")


print("=" * 70)
print("LEXINGTON → FORT WAYNE HISTORICAL DATA")
print("=" * 70)

print(f"Historical observations: {len(route_history):,}")


# ============================================================
# HISTORICAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("HISTORICAL RATE SUMMARY")
print("=" * 70)

display(
    route_history["posted_rate"]
    .describe()
    .to_frame()
    .T
)


# ============================================================
# HISTORICAL ROUTE CHARACTERISTICS
# ============================================================

print("\n" + "=" * 70)
print("ROUTE CHARACTERISTICS")
print("=" * 70)

display(
    route_history[
        [
            "distance",
            "equipment",
            "weight",
            "market_index",
            "quote_signal",
            "posted_rate"
        ]
    ].describe(include="all").T
)


# ============================================================
# EQUIPMENT ON THIS ROUTE
# ============================================================

print("\n" + "=" * 70)
print("EQUIPMENT USED ON THIS ROUTE")
print("=" * 70)

print(route_history["equipment"].value_counts())


# ============================================================
# RECENT ROUTE RATES
# ============================================================

print("\n" + "=" * 70)
print("10 MOST RECENT ROUTE OBSERVATIONS")
print("=" * 70)

display(
    route_history[
        [
            "date",
            "distance",
            "equipment",
            "weight",
            "market_index",
            "quote_signal",
            "posted_rate"
        ]
    ].tail(10)
)


# ============================================================
# MONTHLY ROUTE RATES
# ============================================================

print("\n" + "=" * 70)
print("ROUTE RATE BY MONTH")
print("=" * 70)

route_history["month"] = route_history["date"].dt.month

route_monthly = (
    route_history
    .groupby("month")["posted_rate"]
    .agg(["count", "mean", "median", "min", "max"])
)

display(route_monthly)

LEXINGTON → FORT WAYNE HISTORICAL DATA
Historical observations: 32

HISTORICAL RATE SUMMARY


,count,mean,std,min,25%,50%,75%,max
posted_rate,32.0,856.589062,63.194715,757.93,796.6525,848.61,912.59,973.75



ROUTE CHARACTERISTICS


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
distance,32.0,NaN,NaN,NaN,363.66875,8.854612,346.3,358.15,362.3,368.15,384.5
equipment,32,3,Dry Van,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,32.0,NaN,NaN,NaN,29185.625,7279.155214,16195.0,23955.25,27336.5,36636.0,41138.0
market_index,32.0,NaN,NaN,NaN,1.041472,0.174214,0.79554,0.896825,1.04402,1.154707,1.40617
quote_signal,32.0,NaN,NaN,NaN,2.010673,0.325412,1.55362,1.7526,1.97252,2.2495,2.70631
posted_rate,32.0,NaN,NaN,NaN,856.589062,63.194715,757.93,796.6525,848.61,912.59,973.75



EQUIPMENT USED ON THIS ROUTE
equipment
Dry Van    21
Reefer      8
Flatbed     3
Name: count, dtype: int64

10 MOST RECENT ROUTE OBSERVATIONS


,date,distance,equipment,weight,market_index,quote_signal,posted_rate
35557,2025-08-13,360.8,Dry Van,22007.0,1.06957,1.61911,796.71
36972,2025-08-22,358.6,Reefer,30227.0,1.04508,2.52477,946.18
37239,2025-08-24,384.5,Reefer,24375.0,0.87043,1.55362,935.47
37870,2025-08-27,346.3,Reefer,38755.0,1.05429,1.80401,910.55
40765,2025-09-15,350.9,Dry Van,22972.0,0.81021,2.24980,778.24
41842,2025-09-22,369.8,Reefer,26905.0,0.80501,2.60044,959.92
43271,2025-10-01,371.4,Dry Van,26829.0,0.90458,1.87336,841.42
43376,2025-10-02,354.1,Reefer,27073.0,0.96814,1.58181,901.41
47076,2025-10-26,363.0,Dry Van,17917.0,0.83786,1.95187,796.48
47722,2025-10-30,370.3,Dry Van,26358.0,1.01336,1.82144,864.61



ROUTE RATE BY MONTH


,count,mean,median,min,max
month,,,,,
1,5,811.130000,788.080,757.93,933.53
2,2,830.735000,830.735,791.25,870.22
3,4,845.640000,808.750,791.31,973.75
4,6,861.053333,848.610,786.50,919.76
5,2,915.500000,915.500,896.63,934.37
6,1,824.390000,824.390,824.39,824.39
7,2,879.235000,879.235,877.84,880.63
8,4,897.227500,923.010,796.71,946.18
9,2,869.080000,869.080,778.24,959.92


## 9. Analyze Comparable Historical Loads

The December prediction uses a specific combination of route, equipment, distance, and weight.

We therefore examine historical loads that most closely match the December shipment, particularly Lexington-to-Fort Wayne Dry Van loads.

This provides a more relevant benchmark than using the overall route average.

In [11]:
# ============================================================
# 9. ANALYZE COMPARABLE HISTORICAL LOADS
# ============================================================

# December shipment characteristics
dec_distance = 360
dec_weight = 32000
dec_equipment = "Dry Van"

# Filter to the same route and equipment
comparable = route_history[
    route_history["equipment"] == dec_equipment
].copy()

print("=" * 70)
print("LEXINGTON → FORT WAYNE DRY VAN HISTORY")
print("=" * 70)

print(f"Historical Dry Van observations: {len(comparable):,}")

print("\nRate summary:")
display(
    comparable["posted_rate"]
    .describe()
    .to_frame()
    .T
)


# ============================================================
# DISTANCE COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("DISTANCE COMPARISON")
print("=" * 70)

print(f"December distance: {dec_distance} miles")
print(
    f"Historical Dry Van mean: "
    f"{comparable['distance'].mean():.1f} miles"
)

print(
    f"Historical Dry Van median: "
    f"{comparable['distance'].median():.1f} miles"
)


# ============================================================
# WEIGHT COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("WEIGHT COMPARISON")
print("=" * 70)

print(f"December weight: {dec_weight:,} lb")
print(
    f"Historical Dry Van mean: "
    f"{comparable['weight'].mean():,.0f} lb"
)

print(
    f"Historical Dry Van median: "
    f"{comparable['weight'].median():,.0f} lb"
)


# ============================================================
# HISTORICAL DRY VAN OBSERVATIONS
# ============================================================

print("\n" + "=" * 70)
print("HISTORICAL DRY VAN OBSERVATIONS")
print("=" * 70)

display(
    comparable[
        [
            "date",
            "distance",
            "weight",
            "market_index",
            "quote_signal",
            "posted_rate"
        ]
    ].sort_values("date")
)


# ============================================================
# RECENT DRY VAN RATES
# ============================================================

print("\n" + "=" * 70)
print("RECENT DRY VAN RATES")
print("=" * 70)

display(
    comparable[
        [
            "date",
            "distance",
            "weight",
            "posted_rate"
        ]
    ]
    .sort_values("date")
    .tail(10)
)

LEXINGTON → FORT WAYNE DRY VAN HISTORY
Historical Dry Van observations: 21

Rate summary:


,count,mean,std,min,25%,50%,75%,max
posted_rate,21.0,823.314286,46.151697,757.93,791.25,807.89,849.17,934.37



DISTANCE COMPARISON
December distance: 360 miles
Historical Dry Van mean: 363.8 miles
Historical Dry Van median: 362.5 miles

WEIGHT COMPARISON
December weight: 32,000 lb
Historical Dry Van mean: 28,241 lb
Historical Dry Van median: 26,829 lb

HISTORICAL DRY VAN OBSERVATIONS


,date,distance,weight,market_index,quote_signal,posted_rate
654,2025-01-05,358.3,22567.0,0.79554,2.11108,757.93
1571,2025-01-10,358.2,31556.0,0.87356,2.18657,788.08
2821,2025-01-18,362.1,27842.0,0.84023,2.14650,768.22
3913,2025-01-25,357.5,37102.0,0.92241,2.26175,807.89
6788,2025-02-12,376.8,41021.0,1.04296,2.30284,870.22
8369,2025-02-23,355.1,27601.0,0.95940,2.21525,791.25
10557,2025-03-08,359.9,24178.0,1.01780,2.19709,791.31
11817,2025-03-16,367.6,16195.0,0.99336,2.21027,806.54
12207,2025-03-19,363.1,27600.0,1.14812,2.24940,810.96
14448,2025-04-01,362.8,25383.0,1.07886,1.99317,786.50



RECENT DRY VAN RATES


,date,distance,weight,posted_rate
18777,2025-04-28,366.6,23287.0,848.05
22418,2025-05-22,372.9,35578.0,896.63
23524,2025-05-29,378.8,39179.0,934.37
26302,2025-06-15,359.7,19949.0,824.39
31113,2025-07-15,362.5,36951.0,880.63
35557,2025-08-13,360.8,22007.0,796.71
40765,2025-09-15,350.9,22972.0,778.24
43271,2025-10-01,371.4,26829.0,841.42
47076,2025-10-26,363.0,17917.0,796.48
47722,2025-10-30,370.3,26358.0,864.61


## 10. Build a Time-Based Validation Strategy

The final objective is to predict freight rates for December using historical data from January through October.

To simulate this forecasting problem during development, we use time-based validation rather than a random train/test split.

Data from **January through September** is used for development training, while **October** is held out as a future validation period.

This approach better reflects the real-world forecasting task because the model is trained on earlier freight rates and evaluated on a later month.

In [12]:
# ============================================================
# CLEAN DEVELOPMENT SPLIT
# ============================================================

model_data = pd.read_csv(TRAIN_PATH)

model_data["date"] = pd.to_datetime(model_data["date"])

# January through September
dev_train = model_data[
    model_data["date"] < "2025-10-01"
].copy()

# October
dev_valid = model_data[
    (model_data["date"] >= "2025-10-01") &
    (model_data["date"] < "2025-11-01")
].copy()

print("=" * 70)
print("CLEAN DEVELOPMENT SPLIT")
print("=" * 70)

print(f"Training rows:   {len(dev_train):,}")
print(f"Validation rows: {len(dev_valid):,}")

print(
    f"\nTraining:   {dev_train['date'].min().date()} → "
    f"{dev_train['date'].max().date()}"
)

print(
    f"Validation: {dev_valid['date'].min().date()} → "
    f"{dev_valid['date'].max().date()}"
)

CLEAN DEVELOPMENT SPLIT
Training rows:   43,147
Validation rows: 4,853

Training:   2025-01-01 → 2025-09-30
Validation: 2025-10-01 → 2025-10-31


## 11. Feature Engineering and Missing-Value Handling

We transform the raw shipment data into model-ready features and handle missing shipment weights.

Missing weights are filled using the training-set median, while a missing-value indicator preserves information about which observations originally lacked a weight.

In [13]:
# ============================================================
# 11. FEATURE ENGINEERING + MISSING VALUE HANDLING
# ============================================================

# Calculate the median ONLY from the development training data.
# This prevents information from the validation set leaking into training.
WEIGHT_MEDIAN = dev_train["weight"].median()

print(f"Training weight median: {WEIGHT_MEDIAN:,.2f}")


def create_features(df):
    df = df.copy()

    # --------------------------------------------------------
    # MISSING WEIGHT
    # --------------------------------------------------------

    # Remember whether weight was originally missing
    df["weight_missing"] = df["weight"].isna().astype(int)

    # Fill missing weight using training-set median
    df["weight"] = df["weight"].fillna(WEIGHT_MEDIAN)

    # --------------------------------------------------------
    # DATE FEATURES
    # --------------------------------------------------------

    df["date"] = pd.to_datetime(df["date"])

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_year"] = df["date"].dt.dayofyear
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

    # Cyclical time features
    df["month_sin"] = np.sin(
        2 * np.pi * df["month"] / 12
    )

    df["month_cos"] = np.cos(
        2 * np.pi * df["month"] / 12
    )

    df["dow_sin"] = np.sin(
        2 * np.pi * df["day_of_week"] / 7
    )

    df["dow_cos"] = np.cos(
        2 * np.pi * df["day_of_week"] / 7
    )

    # --------------------------------------------------------
    # ROUTE
    # --------------------------------------------------------

    df["route"] = (
        df["pickup"].astype(str)
        + " → "
        + df["delivery"].astype(str)
    )

    # --------------------------------------------------------
    # WEIGHT FEATURES
    # --------------------------------------------------------

    df["weight_abs"] = df["weight"].abs()

    df["weight_negative"] = (
        df["weight"] < 0
    ).astype(int)

    df["weight_log"] = np.log1p(
        df["weight_abs"]
    )

    # --------------------------------------------------------
    # DISTANCE FEATURES
    # --------------------------------------------------------

    df["distance_squared"] = (
        df["distance"] ** 2
    )

    df["distance_log"] = np.log1p(
        df["distance"]
    )

    # --------------------------------------------------------
    # DISTANCE / WEIGHT RELATIONSHIP
    # --------------------------------------------------------

    df["distance_per_weight"] = (
        df["distance"] /
        df["weight_abs"].clip(lower=1000)
    )

    return df

Training weight median: 31,435.00


In [14]:
dev_train_features = create_features(dev_train)
dev_valid_features = create_features(dev_valid)

In [15]:
print("=" * 70)
print("MISSING VALUES AFTER FEATURE ENGINEERING")
print("=" * 70)

print("Training:")
print(
    dev_train_features.isna().sum()
    [dev_train_features.isna().sum() > 0]
)

print("\nValidation:")
print(
    dev_valid_features.isna().sum()
    [dev_valid_features.isna().sum() > 0]
)

MISSING VALUES AFTER FEATURE ENGINEERING
Training:
market_index    332
dtype: int64

Validation:
market_index    42
dtype: int64


## 12. Prepare Model Features

We select the features that will be provided to the model while removing identifiers, the target variable, and unused columns.

The `market_index` feature is intentionally excluded from training because it contains missing values and we want to establish a clean baseline without imputing this feature. This also allows us to evaluate how well the model performs using the underlying shipment, route, distance, weight, and time information.

Categorical features such as pickup location, delivery location, equipment type, and route are retained so CatBoost can learn route-specific pricing patterns.

In [16]:
# ============================================================
# 12. PREPARE MODEL FEATURES
# ============================================================

DROP_COLUMNS = [
    "load_id",
    "date",
    "posted_rate",
    "market_index"
]

X_train = dev_train_features.drop(
    columns=DROP_COLUMNS
)

X_valid = dev_valid_features.drop(
    columns=DROP_COLUMNS
)

y_train = dev_train_features["posted_rate"]
y_valid = dev_valid_features["posted_rate"]


print("=" * 70)
print("MODEL DATA")
print("=" * 70)

print(f"X_train shape: {X_train.shape}")
print(f"X_valid shape: {X_valid.shape}")

print(f"y_train shape: {y_train.shape}")
print(f"y_valid shape: {y_valid.shape}")

print("\nModel features:")
print(X_train.columns.tolist())

MODEL DATA
X_train shape: (43147, 28)
X_valid shape: (4853, 28)
y_train shape: (43147,)
y_valid shape: (4853,)

Model features:
['pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'quote_signal', 'weight_missing', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'route', 'weight_abs', 'weight_negative', 'weight_log', 'distance_squared', 'distance_log', 'distance_per_weight']


## 13. Define Categorical Features

CatBoost can directly learn from categorical variables without requiring one-hot encoding.

We identify pickup location, delivery location, equipment type, and route as categorical features. These variables allow the model to learn pricing patterns associated with specific locations, equipment types, and recurring routes.

In [17]:
# ============================================================
# 13. CATEGORICAL FEATURES
# ============================================================

categorical_features = [
    "pickup",
    "delivery",
    "equipment",
    "route"
]

print("=" * 70)
print("CATEGORICAL FEATURES")
print("=" * 70)

print(categorical_features)

CATEGORICAL FEATURES
['pickup', 'delivery', 'equipment', 'route']


## 14. Train and Evaluate the CatBoost Baseline

We train a CatBoost regression model using the January–September development data and evaluate it on the October holdout set.

The primary evaluation metric is **Root Mean Squared Error (RMSE)**, which measures prediction error while giving greater weight to large mistakes. We also report **Mean Absolute Error (MAE)** and **R²** to provide additional insight into model performance.

Early stopping is used to prevent unnecessary training once October validation RMSE stops improving.

The resulting October metrics establish our **baseline benchmark**, which future modeling improvements must beat.

In [18]:
# ============================================================
# 14. TRAIN AND EVALUATE CATBOOST BASELINE
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np


# ============================================================
# TRAIN CATBOOST
# ============================================================

print("=" * 70)
print("TRAINING CATBOOST BASELINE")
print("=" * 70)

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=250
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
    early_stopping_rounds=200
)


# ============================================================
# BEST MODEL INFORMATION
# ============================================================

best_iteration = model.get_best_iteration()

best_train_rmse = (
    model.get_best_score()["learn"]["RMSE"]
)

best_rmse = (
    model.get_best_score()["validation"]["RMSE"]
)


print("\n" + "=" * 70)
print("BEST MODEL")
print("=" * 70)

print(f"Best iteration:       {best_iteration}")
print(f"Training RMSE:        ${best_train_rmse:,.2f}")
print(f"Validation RMSE:      ${best_rmse:,.2f}")


# ============================================================
# OCTOBER VALIDATION PREDICTIONS
# ============================================================

valid_predictions = model.predict(X_valid)


# ============================================================
# CALCULATE EVALUATION METRICS
# ============================================================

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        valid_predictions
    )
)

mae = mean_absolute_error(
    y_valid,
    valid_predictions
)

r2 = r2_score(
    y_valid,
    valid_predictions
)


# ============================================================
# FINAL BASELINE RESULTS
# ============================================================

print("\n" + "=" * 70)
print("OCTOBER VALIDATION RESULTS")
print("=" * 70)

print(f"RMSE: ${rmse:,.2f}")
print(f"MAE:  ${mae:,.2f}")
print(f"R²:   {r2:.4f}")


# ============================================================
# BASELINE BENCHMARK
# ============================================================

print("\n" + "=" * 70)
print("BASELINE BENCHMARK")
print("=" * 70)

print(f"October RMSE: ${rmse:,.2f}")
print("Goal: Improve this RMSE in later experiments.")

TRAINING CATBOOST BASELINE
0:	learn: 1446.8473583	test: 1493.8480297	best: 1493.8480297 (0)	total: 307ms	remaining: 15m 20s
250:	learn: 574.0261754	test: 647.5205426	best: 647.5205426 (250)	total: 39s	remaining: 7m 7s
500:	learn: 563.6276040	test: 647.6158919	best: 647.0988323 (352)	total: 1m 12s	remaining: 6m 2s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 647.0988323
bestIteration = 352

Shrink model to first 353 iterations.

BEST MODEL
Best iteration:       352
Training RMSE:        $561.88
Validation RMSE:      $647.10

OCTOBER VALIDATION RESULTS
RMSE: $647.10
MAE:  $117.92
R²:   0.8208

BASELINE BENCHMARK
October RMSE: $647.10
Goal: Improve this RMSE in later experiments.


## 15. Tune and Blend Multiple Gradient-Boosting Models

The CatBoost baseline achieved an October validation RMSE of **$647.10**. Rather than assuming that further feature engineering will improve performance, we now test whether model configuration and model diversity can produce better predictions.

We evaluate three tree-based gradient-boosting approaches:

- **CatBoost** — particularly well suited to the categorical route and equipment features.
- **LightGBM** — provides a different gradient-boosting implementation and tree structure.
- **XGBoost** — provides another independent tree-based modeling approach.

For CatBoost, we test a small range of hyperparameters including tree depth, learning rate, regularization, and sampling parameters. LightGBM and XGBoost are also tested using several configurations.

All models use the same **January–September training data and October holdout set**. RMSE is the primary selection metric because the assessment evaluates the accuracy of freight-rate predictions and penalizes large errors more heavily.

After evaluating the individual models, we blend the strongest predictions using weighted averages. Blending can improve generalization when different models make different prediction errors.

The original CatBoost baseline of **$647.10 RMSE** remains the benchmark. Any improvement must be demonstrated on the October holdout set.

In [19]:
# ============================================================
# 15. MODEL TUNING + MULTI-MODEL BLENDING
# ============================================================

import warnings
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

warnings.filterwarnings("ignore")


# ============================================================
# CONFIGURATION
# ============================================================

BASELINE_RMSE = 647.10

print("=" * 70)
print("MODEL TUNING + MULTI-MODEL BLENDING")
print("=" * 70)

print(f"Baseline RMSE: ${BASELINE_RMSE:,.2f}")
print(f"Training rows: {len(X_train):,}")
print(f"Validation rows: {len(X_valid):,}")


# ============================================================
# STORAGE
# ============================================================

model_results = []
model_predictions = {}


# ============================================================
# METRIC FUNCTION
# ============================================================

def evaluate_model(name, predictions):

    rmse = np.sqrt(
        mean_squared_error(
            y_valid,
            predictions
        )
    )

    mae = mean_absolute_error(
        y_valid,
        predictions
    )

    r2 = r2_score(
        y_valid,
        predictions
    )

    model_results.append({
        "model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    model_predictions[name] = predictions

    print(
        f"{name:<30} "
        f"RMSE: ${rmse:,.2f} | "
        f"MAE: ${mae:,.2f} | "
        f"R²: {r2:.4f}"
    )

    return rmse


# ============================================================
# 1. TUNED CATBOOST MODELS
# ============================================================

print("\n" + "=" * 70)
print("CATBOOST HYPERPARAMETER EXPERIMENTS")
print("=" * 70)


catboost_configs = {

    "CatBoost_Tuned_1": {
        "depth": 6,
        "learning_rate": 0.03,
        "l2_leaf_reg": 5,
        "random_strength": 1,
        "bagging_temperature": 1
    },

    "CatBoost_Tuned_2": {
        "depth": 7,
        "learning_rate": 0.03,
        "l2_leaf_reg": 8,
        "random_strength": 1,
        "bagging_temperature": 1
    },

    "CatBoost_Tuned_3": {
        "depth": 8,
        "learning_rate": 0.02,
        "l2_leaf_reg": 10,
        "random_strength": 0.5,
        "bagging_temperature": 1
    }
}


for name, params in catboost_configs.items():

    print("\n" + "-" * 70)
    print(name)
    print(params)
    print("-" * 70)

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",

        iterations=5000,

        random_seed=42,

        verbose=500,

        **params
    )

    model.fit(
        X_train,
        y_train,

        cat_features=categorical_features,

        eval_set=(X_valid, y_valid),

        use_best_model=True,

        early_stopping_rounds=300
    )

    predictions = model.predict(X_valid)

    evaluate_model(
        name,
        predictions
    )


# ============================================================
# 2. LIGHTGBM
# ============================================================

print("\n" + "=" * 70)
print("LIGHTGBM EXPERIMENTS")
print("=" * 70)


# ------------------------------------------------------------
# LightGBM requires categorical columns to be encoded
# ------------------------------------------------------------

try:

    import lightgbm as lgb

    X_train_lgb = X_train.copy()
    X_valid_lgb = X_valid.copy()

    lgb_categories = {}

    for col in categorical_features:

        combined = pd.concat(
            [
                X_train_lgb[col],
                X_valid_lgb[col]
            ],
            axis=0
        ).astype(str)

        categories = pd.Categorical(
            combined
        ).categories

        X_train_lgb[col] = pd.Categorical(
            X_train_lgb[col].astype(str),
            categories=categories
        )

        X_valid_lgb[col] = pd.Categorical(
            X_valid_lgb[col].astype(str),
            categories=categories
        )

        lgb_categories[col] = categories


    lgb_configs = {

        "LightGBM_1": {
            "num_leaves": 31,
            "max_depth": 7,
            "learning_rate": 0.03,
            "min_child_samples": 30,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.5,
            "reg_lambda": 2
        },

        "LightGBM_2": {
            "num_leaves": 63,
            "max_depth": 9,
            "learning_rate": 0.02,
            "min_child_samples": 40,
            "subsample": 0.85,
            "colsample_bytree": 0.75,
            "reg_alpha": 1,
            "reg_lambda": 3
        }
    }


    for name, params in lgb_configs.items():

        print("\n" + "-" * 70)
        print(name)
        print(params)
        print("-" * 70)

        model = lgb.LGBMRegressor(
            objective="regression",
            n_estimators=5000,

            random_state=42,

            n_jobs=-1,

            verbosity=-1,

            **params
        )

        model.fit(
            X_train_lgb,
            y_train,

            categorical_feature=categorical_features,

            eval_set=[
                (X_valid_lgb, y_valid)
            ],

            eval_metric="rmse",

            callbacks=[
                lgb.early_stopping(
                    300,
                    verbose=True
                )
            ]
        )

        predictions = model.predict(
            X_valid_lgb,
            num_iteration=model.best_iteration_
        )

        evaluate_model(
            name,
            predictions
        )


except ImportError:

    print(
        "\nLightGBM is not installed. "
        "Skipping LightGBM experiments."
    )


# ============================================================
# 3. XGBOOST
# ============================================================

print("\n" + "=" * 70)
print("XGBOOST EXPERIMENTS")
print("=" * 70)


try:

    import xgboost as xgb


    # --------------------------------------------------------
    # XGBoost cannot directly use our string categoricals.
    # Encode them consistently between train and validation.
    # --------------------------------------------------------

    X_train_xgb = X_train.copy()
    X_valid_xgb = X_valid.copy()

    for col in categorical_features:

        categories = pd.concat(
            [
                X_train_xgb[col],
                X_valid_xgb[col]
            ],
            axis=0
        ).astype(str).unique()

        mapping = {
            value: index
            for index, value in enumerate(categories)
        }

        X_train_xgb[col] = (
            X_train_xgb[col]
            .astype(str)
            .map(mapping)
        )

        X_valid_xgb[col] = (
            X_valid_xgb[col]
            .astype(str)
            .map(mapping)
            .fillna(-1)
        )


    xgb_configs = {

        "XGBoost_1": {
            "max_depth": 6,
            "learning_rate": 0.03,
            "min_child_weight": 5,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.5,
            "reg_lambda": 2
        },

        "XGBoost_2": {
            "max_depth": 8,
            "learning_rate": 0.02,
            "min_child_weight": 8,
            "subsample": 0.85,
            "colsample_bytree": 0.75,
            "reg_alpha": 1,
            "reg_lambda": 3
        }
    }


    for name, params in xgb_configs.items():

        print("\n" + "-" * 70)
        print(name)
        print(params)
        print("-" * 70)

        model = xgb.XGBRegressor(
            objective="reg:squarederror",

            n_estimators=5000,

            random_state=42,

            n_jobs=-1,

            tree_method="hist",

            **params
        )

        model.fit(
            X_train_xgb,
            y_train,

            eval_set=[
                (X_valid_xgb, y_valid)
            ],

            verbose=500
        )

        predictions = model.predict(
            X_valid_xgb
        )

        evaluate_model(
            name,
            predictions
        )


except ImportError:

    print(
        "\nXGBoost is not installed. "
        "Skipping XGBoost experiments."
    )


# ============================================================
# 4. INDIVIDUAL MODEL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("INDIVIDUAL MODEL RESULTS")
print("=" * 70)

results_df = (
    pd.DataFrame(model_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print(
    results_df.to_string(
        index=False,
        formatters={
            "RMSE": "${:,.2f}".format,
            "MAE": "${:,.2f}".format,
            "R2": "{:.4f}".format
        }
    )
)


# ============================================================
# 5. SELECT TOP MODELS
# ============================================================

print("\n" + "=" * 70)
print("TOP MODELS")
print("=" * 70)

top_models = (
    results_df
    .head(3)["model"]
    .tolist()
)

print(top_models)


# ============================================================
# 6. TEST BLENDS
# ============================================================

print("\n" + "=" * 70)
print("MODEL BLENDING")
print("=" * 70)


best_blend_rmse = np.inf
best_blend_name = None
best_blend_predictions = None


# ------------------------------------------------------------
# Pairwise 50/50 blends
# ------------------------------------------------------------

for i in range(len(top_models)):

    for j in range(i + 1, len(top_models)):

        model_a = top_models[i]
        model_b = top_models[j]

        blend = (
            0.50 * model_predictions[model_a]
            +
            0.50 * model_predictions[model_b]
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                blend
            )
        )

        name = (
            f"Blend_50_50_"
            f"{model_a}_"
            f"{model_b}"
        )

        print(
            f"{name:<70} "
            f"RMSE: ${rmse:,.2f}"
        )

        if rmse < best_blend_rmse:

            best_blend_rmse = rmse
            best_blend_name = name
            best_blend_predictions = blend


# ------------------------------------------------------------
# Weighted blends
# ------------------------------------------------------------

if len(top_models) >= 2:

    model_a = top_models[0]
    model_b = top_models[1]

    weights = [
        0.25,
        0.40,
        0.50,
        0.60,
        0.75
    ]

    print("\nWeighted blend search:")

    for weight in weights:

        blend = (
            weight * model_predictions[model_a]
            +
            (1 - weight)
            * model_predictions[model_b]
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_valid,
                blend
            )
        )

        print(
            f"{weight:.0%} {model_a} + "
            f"{1-weight:.0%} {model_b}"
            f" → RMSE: ${rmse:,.2f}"
        )

        if rmse < best_blend_rmse:

            best_blend_rmse = rmse

            best_blend_name = (
                f"{weight:.0%} {model_a} + "
                f"{1-weight:.0%} {model_b}"
            )

            best_blend_predictions = blend


# ============================================================
# 7. FINAL COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)

best_individual = (
    results_df.iloc[0]
)

print(
    f"Baseline CatBoost RMSE: "
    f"${BASELINE_RMSE:,.2f}"
)

print(
    f"Best individual model: "
    f"{best_individual['model']}"
)

print(
    f"Best individual RMSE: "
    f"${best_individual['RMSE']:,.2f}"
)

if best_blend_name is not None:

    print(
        f"Best blend: "
        f"{best_blend_name}"
    )

    print(
        f"Best blend RMSE: "
        f"${best_blend_rmse:,.2f}"
    )

    if best_blend_rmse < best_individual["RMSE"]:

        final_rmse = best_blend_rmse
        final_predictions = best_blend_predictions
        final_model_name = best_blend_name

    else:

        final_rmse = best_individual["RMSE"]
        final_predictions = model_predictions[
            best_individual["model"]
        ]
        final_model_name = best_individual["model"]

else:

    final_rmse = best_individual["RMSE"]

    final_predictions = model_predictions[
        best_individual["model"]
    ]

    final_model_name = best_individual["model"]


print("\n" + "=" * 70)
print("WINNING DEVELOPMENT MODEL")
print("=" * 70)

print(
    f"Model: {final_model_name}"
)

print(
    f"October RMSE: ${final_rmse:,.2f}"
)

print(
    f"Baseline RMSE: ${BASELINE_RMSE:,.2f}"
)

print(
    f"RMSE change: "
    f"${BASELINE_RMSE - final_rmse:+,.2f}"
)

if final_rmse < BASELINE_RMSE:

    improvement = (
        (BASELINE_RMSE - final_rmse)
        / BASELINE_RMSE
        * 100
    )

    print(
        f"Improvement: "
        f"{improvement:.2f}%"
    )

    print(
        "RESULT: NEW MODEL BEATS BASELINE."
    )

else:

    deterioration = (
        (final_rmse - BASELINE_RMSE)
        / BASELINE_RMSE
        * 100
    )

    print(
        f"Deterioration: "
        f"{deterioration:.2f}%"
    )

    print(
        "RESULT: BASELINE REMAINS STRONGER."
    )


# ============================================================
# 8. FINAL METRICS
# ============================================================

final_mae = mean_absolute_error(
    y_valid,
    final_predictions
)

final_r2 = r2_score(
    y_valid,
    final_predictions
)

print("\n" + "=" * 70)
print("WINNING MODEL METRICS")
print("=" * 70)

print(
    f"RMSE: ${final_rmse:,.2f}"
)

print(
    f"MAE:  ${final_mae:,.2f}"
)

print(
    f"R²:   {final_r2:.4f}"
)


MODEL TUNING + MULTI-MODEL BLENDING
Baseline RMSE: $647.10
Training rows: 43,147
Validation rows: 4,853

CATBOOST HYPERPARAMETER EXPERIMENTS

----------------------------------------------------------------------
CatBoost_Tuned_1
{'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 1}
----------------------------------------------------------------------
0:	learn: 1447.1085076	test: 1493.9226842	best: 1493.9226842 (0)	total: 176ms	remaining: 14m 41s
500:	learn: 573.0153461	test: 647.7001012	best: 647.2463688 (414)	total: 33.2s	remaining: 4m 58s
1000:	learn: 563.1901700	test: 648.1484642	best: 646.9761276 (722)	total: 1m 7s	remaining: 4m 29s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 646.9761276
bestIteration = 722

Shrink model to first 723 iterations.
CatBoost_Tuned_1               RMSE: $646.98 | MAE: $124.54 | R²: 0.8209

----------------------------------------------------------------------
CatBoost_Tuned_2
{'de

## 16. Retrain the Winning Models and Generate Final Predictions

After comparing multiple models and hyperparameter configurations, the strongest development approach was a **50/50 blend of two CatBoost models**:

- **CatBoost Tuned 1** — depth 6, learning rate 0.03, L2 regularization of 5.
- **CatBoost Tuned 3** — depth 8, learning rate 0.02, L2 regularization of 10.

The blended model achieved an October holdout RMSE of $646.51, improving on the original CatBoost baseline of $647.10.

For the final prediction stage, both winning models are retrained using **100% of the labeled development data in `train-test.csv`**, allowing them to learn from all available labeled observations.

The same feature engineering and preprocessing pipeline is applied across the development, validation, and December datasets. The final feature set is restricted to features available across all three datasets, ensuring the models can generate valid December predictions.

The final predictions use the same **50/50 CatBoost blend** and produce:

1. `validation_predictions.csv` — predictions for all 12,000 validation loads.
2. `december_chart_inputs.csv` — the December chart inputs with `predicted_rate` added for all 31 days.

This final retraining step allows the selected models to use all available labeled data while preserving the modeling approach established during validation.

In [22]:
# ============================================================
# 16. RETRAIN WINNING MODELS AND GENERATE FINAL PREDICTIONS
# ============================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")


# ============================================================
# CONFIGURATION
# ============================================================

# Repository structure:
#
# Freight-Rate-Prediction/
# │
# ├── freight-rate-prediction.ipynb
# ├── README.md
# ├── requirements.txt
# │
# ├── data/
# │   ├── train-test.csv
# │   ├── validation.csv
# │   ├── december-chart-inputs.csv
# │   └── validation-predictions-template.csv
# │
# └── outputs/
#     ├── validation_predictions.csv
#     └── december_chart_inputs.csv
#
# Relative paths allow the notebook to run locally
# without depending on Kaggle-specific paths.


DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./outputs")


# ============================================================
# INPUT FILE PATHS
# ============================================================

TRAIN_PATH = DATA_DIR / "train-test.csv"

VALIDATION_PATH = DATA_DIR / "validation.csv"

DECEMBER_PATH = DATA_DIR / "december-chart-inputs.csv"

TEMPLATE_PATH = DATA_DIR / "validation-predictions-template.csv"


# ============================================================
# OUTPUT FILE PATHS
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VALIDATION_OUTPUT = (
    OUTPUT_DIR / "validation_predictions.csv"
)

DECEMBER_OUTPUT = (
    OUTPUT_DIR / "december_chart_inputs.csv"
)


# ============================================================
# VERIFY REQUIRED FILES
# ============================================================

print("=" * 70)
print("CHECKING REQUIRED DATA FILES")
print("=" * 70)

required_files = {
    "Training data": TRAIN_PATH,
    "Validation data": VALIDATION_PATH,
    "December data": DECEMBER_PATH,
    "Validation template": TEMPLATE_PATH
}

for name, path in required_files.items():

    if not path.is_file():

        raise FileNotFoundError(
            f"\n{name} not found at:\n"
            f"{path.resolve()}\n\n"
            "Make sure the required files are inside "
            "the repository's 'data' folder."
        )

    print(
        f"{name}: OK"
    )


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING FINAL DATASETS")
print("=" * 70)

train = pd.read_csv(
    TRAIN_PATH
)

validation = pd.read_csv(
    VALIDATION_PATH
)

december = pd.read_csv(
    DECEMBER_PATH
)

template = pd.read_csv(
    TEMPLATE_PATH
)

print(
    f"Training data:   {train.shape}"
)

print(
    f"Validation data: {validation.shape}"
)

print(
    f"December data:   {december.shape}"
)

print(
    f"Template:        {template.shape}"
)


# ============================================================
# BASIC DATA CHECKS
# ============================================================

print("\n" + "=" * 70)
print("BASIC DATA CHECKS")
print("=" * 70)

print("\nTraining columns:")
print(train.columns.tolist())

print("\nValidation columns:")
print(validation.columns.tolist())

print("\nDecember columns:")
print(december.columns.tolist())

print("\nTemplate columns:")
print(template.columns.tolist())


# ============================================================
# IDENTIFY TARGET
# ============================================================

TARGET_CANDIDATES = [
    "posted_rate",
    "rate",
    "freight_rate",
    "target",
    "actual_rate",
    "predicted_rate"
]

target_candidates_found = [
    col
    for col in TARGET_CANDIDATES
    if col in train.columns
]

if len(target_candidates_found) == 0:

    raise ValueError(
        "Could not identify the target column. "
        f"Training columns are: {train.columns.tolist()}"
    )

TARGET = target_candidates_found[0]

print("\n" + "=" * 70)
print("TARGET")
print("=" * 70)

print(
    f"Target column: {TARGET}"
)


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def engineer_features(df):

    df = df.copy()


    # --------------------------------------------------------
    # DATE FEATURES
    # --------------------------------------------------------

    if "date" in df.columns:

        date = pd.to_datetime(
            df["date"],
            errors="coerce"
        )

        df["year"] = date.dt.year

        df["month"] = date.dt.month

        df["day"] = date.dt.day

        df["day_of_week"] = date.dt.dayofweek

        df["week_of_year"] = (
            date.dt.isocalendar()
            .week
            .astype(float)
        )

        df["day_of_year"] = date.dt.dayofyear


        # Cyclical month features

        df["month_sin"] = np.sin(
            2 * np.pi * df["month"] / 12
        )

        df["month_cos"] = np.cos(
            2 * np.pi * df["month"] / 12
        )


        # Cyclical day-of-week features

        df["dow_sin"] = np.sin(
            2 * np.pi * df["day_of_week"] / 7
        )

        df["dow_cos"] = np.cos(
            2 * np.pi * df["day_of_week"] / 7
        )


        # Remove raw date

        df = df.drop(
            columns=["date"]
        )


    # --------------------------------------------------------
    # DISTANCE FEATURES
    # --------------------------------------------------------

    if "distance" in df.columns:

        distance = pd.to_numeric(
            df["distance"],
            errors="coerce"
        )

        df["distance_squared"] = (
            distance ** 2
        )

        df["distance_cubed"] = (
            distance ** 3
        )

        df["distance_log"] = np.log1p(
            distance.clip(lower=0)
        )

        df["distance_sqrt"] = np.sqrt(
            distance.clip(lower=0)
        )


    # --------------------------------------------------------
    # WEIGHT FEATURES
    # --------------------------------------------------------

    if "weight" in df.columns:

        weight = pd.to_numeric(
            df["weight"],
            errors="coerce"
        )

        df["weight_squared"] = (
            weight ** 2
        )

        df["weight_log"] = np.log1p(
            weight.clip(lower=0)
        )

        df["weight_abs"] = (
            weight.abs()
        )


    # --------------------------------------------------------
    # DISTANCE / WEIGHT RELATIONSHIPS
    # --------------------------------------------------------

    if (
        "distance" in df.columns
        and "weight" in df.columns
    ):

        distance = pd.to_numeric(
            df["distance"],
            errors="coerce"
        )

        weight = pd.to_numeric(
            df["weight"],
            errors="coerce"
        )

        df["distance_per_weight"] = (
            distance /
            (weight + 1)
        )

        df["distance_weight_product"] = (
            distance * weight
        )


    # --------------------------------------------------------
    # LOCATION FEATURES
    # --------------------------------------------------------

    if (
        "pickup_lat" in df.columns
        and "delivery_lat" in df.columns
    ):

        df["lat_difference"] = (
            df["delivery_lat"]
            - df["pickup_lat"]
        )

        df["lat_difference_abs"] = (
            df["lat_difference"]
            .abs()
        )


    if (
        "pickup_lon" in df.columns
        and "delivery_lon" in df.columns
    ):

        df["lon_difference"] = (
            df["delivery_lon"]
            - df["pickup_lon"]
        )

        df["lon_difference_abs"] = (
            df["lon_difference"]
            .abs()
        )


    # --------------------------------------------------------
    # ROUTE FEATURES
    # --------------------------------------------------------

    if (
        "pickup" in df.columns
        and "delivery" in df.columns
    ):

        df["route"] = (
            df["pickup"].astype(str)
            + "_"
            + df["delivery"].astype(str)
        )

        df["route_pair"] = (
            df["pickup"].astype(str)
            + " → "
            + df["delivery"].astype(str)
        )


    # --------------------------------------------------------
    # EQUIPMENT INTERACTIONS
    # --------------------------------------------------------

    if (
        "pickup" in df.columns
        and "equipment" in df.columns
    ):

        df["pickup_equipment"] = (
            df["pickup"].astype(str)
            + "_"
            + df["equipment"].astype(str)
        )


    if (
        "delivery" in df.columns
        and "equipment" in df.columns
    ):

        df["delivery_equipment"] = (
            df["delivery"].astype(str)
            + "_"
            + df["equipment"].astype(str)
        )


    if (
        "route" in df.columns
        and "equipment" in df.columns
    ):

        df["route_equipment"] = (
            df["route"].astype(str)
            + "_"
            + df["equipment"].astype(str)
        )


    # --------------------------------------------------------
    # ROUTE + TIME
    # --------------------------------------------------------

    if (
        "route" in df.columns
        and "month" in df.columns
    ):

        df["route_month"] = (
            df["route"].astype(str)
            + "_"
            + df["month"].astype(str)
        )


    if (
        "equipment" in df.columns
        and "month" in df.columns
    ):

        df["equipment_month"] = (
            df["equipment"].astype(str)
            + "_"
            + df["month"].astype(str)
        )


    # --------------------------------------------------------
    # MARKET INTERACTIONS
    # --------------------------------------------------------

    if (
        "market_index" in df.columns
        and "distance" in df.columns
    ):

        df["market_distance_interaction"] = (
            df["market_index"]
            * df["distance"]
        )


    if (
        "market_index" in df.columns
        and "quote_signal" in df.columns
    ):

        df["market_quote_interaction"] = (
            df["market_index"]
            * df["quote_signal"]
        )


    if (
        "quote_signal" in df.columns
        and "distance" in df.columns
    ):

        df["quote_distance_interaction"] = (
            df["quote_signal"]
            * df["distance"]
        )


    # --------------------------------------------------------
    # CLEAN INFINITE VALUES
    # --------------------------------------------------------

    df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    return df


# ============================================================
# CREATE RAW FEATURES
# ============================================================

print("\n" + "=" * 70)
print("CREATING FINAL FEATURES")
print("=" * 70)


X_full_raw = train.drop(
    columns=[TARGET],
    errors="ignore"
).copy()


X_validation_raw = validation.drop(
    columns=["load_id"],
    errors="ignore"
).copy()


# December already contains predicted_rate as a
# placeholder/output column. It must not be used
# as a model feature.

X_december_raw = december.drop(
    columns=["predicted_rate"],
    errors="ignore"
).copy()


# ============================================================
# ENGINEER FEATURES
# ============================================================

X_full = engineer_features(
    X_full_raw
)

X_validation = engineer_features(
    X_validation_raw
)

X_december = engineer_features(
    X_december_raw
)


print(
    f"Development features: {X_full.shape}"
)

print(
    f"Validation features:  {X_validation.shape}"
)

print(
    f"December features:    {X_december.shape}"
)


# ============================================================
# ALIGN FEATURE COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("ALIGNING FINAL MODEL FEATURES")
print("=" * 70)


# December contains fewer raw columns.
#
# Therefore, the production model uses only features
# available in all three datasets.

common_features = sorted(
    set(X_full.columns)
    & set(X_validation.columns)
    & set(X_december.columns)
)


if len(common_features) == 0:

    raise ValueError(
        "No common features exist across "
        "training, validation and December data."
    )


X_full = X_full[
    common_features
].copy()

X_validation = X_validation[
    common_features
].copy()

X_december = X_december[
    common_features
].copy()


print(
    f"Common model features: "
    f"{len(common_features)}"
)


print("\nCommon features:")

for feature in common_features:

    print(
        f"  - {feature}"
    )


# ============================================================
# TARGET
# ============================================================

y_full = pd.to_numeric(
    train[TARGET],
    errors="coerce"
)


if y_full.isna().any():

    raise ValueError(
        "Target contains missing or invalid values."
    )


print("\nTarget statistics:")

print(
    y_full.describe()
)


# ============================================================
# CATEGORICAL FEATURES
# ============================================================

categorical_features = [

    "pickup",
    "delivery",
    "equipment",

    "route",
    "route_pair",

    "pickup_equipment",
    "delivery_equipment",
    "route_equipment",

    "route_month",
    "equipment_month"

]


categorical_features = [
    col
    for col in categorical_features
    if col in X_full.columns
]


print("\n" + "=" * 70)
print("CATEGORICAL FEATURES")
print("=" * 70)

print(
    categorical_features
)


# ============================================================
# MAKE CATEGORICAL VALUES CONSISTENT
# ============================================================

for col in categorical_features:

    X_full[col] = (
        X_full[col]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_validation[col] = (
        X_validation[col]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_december[col] = (
        X_december[col]
        .fillna("__MISSING__")
        .astype(str)
    )


# ============================================================
# NUMERIC FEATURES
# ============================================================

numeric_features = [
    col
    for col in X_full.columns
    if col not in categorical_features
]


print("\n" + "=" * 70)
print("NUMERIC FEATURES")
print("=" * 70)

print(
    f"Numeric feature count: "
    f"{len(numeric_features)}"
)


# ============================================================
# NUMERIC MISSING VALUES
# ============================================================

# Calculate medians ONLY from labeled development data.

numeric_medians = (
    X_full[numeric_features]
    .median()
)


for df in [
    X_full,
    X_validation,
    X_december
]:

    df[numeric_features] = (
        df[numeric_features]

        .replace(
            [np.inf, -np.inf],
            np.nan
        )

        .fillna(
            numeric_medians
        )
    )


# ============================================================
# FINAL MISSING VALUE CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL MISSING VALUE CHECK")
print("=" * 70)

print(
    "Development NaNs:",
    X_full.isna().sum().sum()
)

print(
    "Validation NaNs:",
    X_validation.isna().sum().sum()
)

print(
    "December NaNs:",
    X_december.isna().sum().sum()
)


# ============================================================
# WINNING CATBOOST MODELS
# ============================================================

models = [

    (
        "CatBoost_Tuned_1",
        {
            "depth": 6,
            "learning_rate": 0.03,
            "l2_leaf_reg": 5,
            "random_strength": 1,
            "bagging_temperature": 1,
            "iterations": 723
        }
    ),

    (
        "CatBoost_Tuned_3",
        {
            "depth": 8,
            "learning_rate": 0.02,
            "l2_leaf_reg": 10,
            "random_strength": 0.5,
            "bagging_temperature": 1,
            "iterations": 547
        }
    )

]


# ============================================================
# TRAIN FINAL MODELS
# ============================================================

print("\n" + "=" * 70)
print("TRAINING FINAL MODELS")
print("=" * 70)


validation_predictions = []

december_predictions = []

trained_models = {}


for model_name, params in models:

    print("\n" + "=" * 70)

    print(
        f"TRAINING FINAL {model_name}"
    )

    print("=" * 70)

    print(
        f"Parameters: {params}"
    )


    # --------------------------------------------------------
    # CREATE MODEL
    # --------------------------------------------------------

    final_model = CatBoostRegressor(

        loss_function="RMSE",

        iterations=params["iterations"],

        learning_rate=params["learning_rate"],

        depth=params["depth"],

        l2_leaf_reg=params["l2_leaf_reg"],

        random_strength=params["random_strength"],

        bagging_temperature=params["bagging_temperature"],

        random_seed=42,

        verbose=100

    )


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    final_model.fit(

        X_full,

        y_full,

        cat_features=categorical_features

    )


    trained_models[
        model_name
    ] = final_model


    # --------------------------------------------------------
    # VALIDATION PREDICTIONS
    # --------------------------------------------------------

    validation_pred = (
        final_model.predict(
            X_validation
        )
    )


    validation_predictions.append(
        validation_pred
    )


    # --------------------------------------------------------
    # DECEMBER PREDICTIONS
    # --------------------------------------------------------

    december_pred = (
        final_model.predict(
            X_december
        )
    )


    december_predictions.append(
        december_pred
    )


    # --------------------------------------------------------
    # MODEL PREDICTION SUMMARY
    # --------------------------------------------------------

    print(
        f"\n{model_name} validation range: "
        f"${validation_pred.min():,.2f} "
        f"to "
        f"${validation_pred.max():,.2f}"
    )

    print(
        f"{model_name} December range: "
        f"${december_pred.min():,.2f} "
        f"to "
        f"${december_pred.max():,.2f}"
    )


# ============================================================
# 50/50 WINNING BLEND
# ============================================================

print("\n" + "=" * 70)
print("CREATING 50/50 WINNING BLEND")
print("=" * 70)


validation_prediction = (
    0.50 * validation_predictions[0]
    +
    0.50 * validation_predictions[1]
)


december_prediction = (
    0.50 * december_predictions[0]
    +
    0.50 * december_predictions[1]
)


print("Blend weights:")

print(
    "CatBoost_Tuned_1: 50%"
)

print(
    "CatBoost_Tuned_3: 50%"
)


# ============================================================
# SANITY CHECK PREDICTIONS
# ============================================================

print("\n" + "=" * 70)
print("PREDICTION SANITY CHECK")
print("=" * 70)


print(
    f"Validation predictions: "
    f"{len(validation_prediction):,}"
)

print(
    f"Expected validation rows: "
    f"{len(validation):,}"
)

print(
    f"December predictions: "
    f"{len(december_prediction):,}"
)

print(
    f"Expected December rows: "
    f"{len(december):,}"
)


print(
    f"\nValidation prediction range: "
    f"${validation_prediction.min():,.2f} "
    f"to "
    f"${validation_prediction.max():,.2f}"
)

print(
    f"December prediction range: "
    f"${december_prediction.min():,.2f} "
    f"to "
    f"${december_prediction.max():,.2f}"
)


# ============================================================
# CHECK PREDICTION COUNTS
# ============================================================

if len(validation_prediction) != len(validation):

    raise ValueError(
        "Validation prediction count does not "
        "match validation rows."
    )


if len(december_prediction) != len(december):

    raise ValueError(
        "December prediction count does not "
        "match December rows."
    )


# ============================================================
# CHECK PREDICTION VALUES
# ============================================================

if (
    ~np.isfinite(validation_prediction)
).any():

    raise ValueError(
        "Validation predictions contain "
        "invalid numeric values."
    )


if (
    ~np.isfinite(december_prediction)
).any():

    raise ValueError(
        "December predictions contain "
        "invalid numeric values."
    )


if (
    validation_prediction <= 0
).any():

    raise ValueError(
        "Validation predictions contain "
        "non-positive values."
    )


if (
    december_prediction <= 0
).any():

    raise ValueError(
        "December predictions contain "
        "non-positive values."
    )


# ============================================================
# CREATE VALIDATION SUBMISSION
# ============================================================

print("\n" + "=" * 70)
print("CREATING validation_predictions.csv")
print("=" * 70)


validation_output = template.copy()


# ------------------------------------------------------------
# CHECK TEMPLATE
# ------------------------------------------------------------

if list(template.columns) != [
    "load_id",
    "predicted_rate"
]:

    raise ValueError(
        "Validation template must contain exactly "
        "load_id,predicted_rate."
    )


if len(validation_output) != len(
    validation_prediction
):

    raise ValueError(
        "Prediction count does not match "
        "validation template."
    )


# ------------------------------------------------------------
# CHECK LOAD IDs
# ------------------------------------------------------------

if "load_id" not in validation.columns:

    raise ValueError(
        "validation.csv does not contain load_id."
    )


if "load_id" not in template.columns:

    raise ValueError(
        "Validation template does not contain load_id."
    )


if not np.array_equal(
    template["load_id"].values,
    validation["load_id"].values
):

    raise ValueError(
        "Validation load_id ordering does not "
        "match the template."
    )


# ------------------------------------------------------------
# FILL PREDICTIONS
# ------------------------------------------------------------

validation_output[
    "predicted_rate"
] = validation_prediction


# ------------------------------------------------------------
# SAVE VALIDATION PREDICTIONS
# ------------------------------------------------------------

validation_output.to_csv(
    VALIDATION_OUTPUT,
    index=False
)


print(
    f"Saved: {VALIDATION_OUTPUT.resolve()}"
)


# ============================================================
# CREATE DECEMBER PREDICTION FILE
# ============================================================

print("\n" + "=" * 70)
print("CREATING DECEMBER CHART INPUT FILE")
print("=" * 70)


# Preserve the original December input columns
# and add the predicted_rate column.

december_output = december.copy()


december_output[
    "predicted_rate"
] = december_prediction


# ============================================================
# CHECK DECEMBER FORMAT
# ============================================================

expected_december_columns = [

    "pickup",
    "delivery",
    "distance",
    "equipment",
    "weight",
    "date",
    "predicted_rate"

]


if list(december_output.columns) != (
    expected_december_columns
):

    raise ValueError(

        "December output columns do not match "
        "the required score.py format.\n"

        f"Expected: "
        f"{expected_december_columns}\n"

        f"Got: "
        f"{december_output.columns.tolist()}"

    )


# ============================================================
# CHECK DECEMBER ROW COUNT
# ============================================================

if len(december_output) != 31:

    raise ValueError(
        "December input must contain exactly "
        "31 rows."
    )


# ============================================================
# CHECK DECEMBER DATES
# ============================================================

december_dates = pd.to_datetime(
    december_output["date"],
    errors="coerce"
)


if december_dates.isna().any():

    raise ValueError(
        "December data contains invalid dates."
    )


expected_dates = pd.date_range(
    "2025-12-01",
    "2025-12-31",
    freq="D"
)


if set(december_dates) != set(
    expected_dates
):

    raise ValueError(
        "December data must contain every date "
        "from 2025-12-01 through 2025-12-31."
    )


# ============================================================
# CHECK FIXED DECEMBER INPUTS
# ============================================================

if not december_output[
    "pickup"
].eq("Lexington").all():

    raise ValueError(
        "December pickup must be Lexington."
    )


if not december_output[
    "delivery"
].eq("Fort Wayne").all():

    raise ValueError(
        "December delivery must be Fort Wayne."
    )


if not np.isclose(
    december_output["distance"].astype(float),
    360.0
).all():

    raise ValueError(
        "December distance must be 360."
    )


if not december_output[
    "equipment"
].eq("Dry Van").all():

    raise ValueError(
        "December equipment must be Dry Van."
    )


if not np.isclose(
    december_output["weight"].astype(float),
    32000.0
).all():

    raise ValueError(
        "December weight must be 32000."
    )


# ============================================================
# SAVE DECEMBER FILE
# ============================================================

december_output.to_csv(
    DECEMBER_OUTPUT,
    index=False
)


print(
    f"Saved: {DECEMBER_OUTPUT.resolve()}"
)


# ============================================================
# FINAL OUTPUT VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL OUTPUT VERIFICATION")
print("=" * 70)


print(
    f"Validation predictions rows: "
    f"{len(validation_output):,}"
)


print(
    f"December prediction rows: "
    f"{len(december_output):,}"
)


print(
    "\nValidation prediction missing values:",
    validation_output[
        "predicted_rate"
    ].isna().sum()
)


print(
    "December prediction missing values:",
    december_output[
        "predicted_rate"
    ].isna().sum()
)


# ============================================================
# FINAL FILE FORMAT CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL FORMAT CHECK")
print("=" * 70)


print(
    "Validation columns:",
    validation_output.columns.tolist()
)


print(
    "December columns:",
    december_output.columns.tolist()
)


if list(validation_output.columns) != [
    "load_id",
    "predicted_rate"
]:

    raise ValueError(
        "Final validation file has incorrect columns."
    )


if list(december_output.columns) != [
    "pickup",
    "delivery",
    "distance",
    "equipment",
    "weight",
    "date",
    "predicted_rate"
]:

    raise ValueError(
        "Final December file has incorrect columns."
    )


# ============================================================
# DISPLAY OUTPUTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL FILES")
print("=" * 70)


print(
    f"Validation file:\n"
    f"  {VALIDATION_OUTPUT.resolve()}"
)


print(
    f"December file:\n"
    f"  {DECEMBER_OUTPUT.resolve()}"
)


print("\nFirst validation predictions:")

print(
    validation_output.head(10)
)


print("\nFirst December predictions:")

print(
    december_output.head(10)
)


# ============================================================
# COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("FINAL PREDICTION PIPELINE COMPLETE")
print("=" * 70)

print(
    "\nYour two files are ready:"
)

print(
    f"1. {VALIDATION_OUTPUT.resolve()}"
)

print(
    f"2. {DECEMBER_OUTPUT.resolve()}"
)

CHECKING REQUIRED DATA FILES
Training data: OK
Validation data: OK
December data: OK
Validation template: OK

LOADING FINAL DATASETS
Training data:   (48000, 14)
Validation data: (12000, 13)
December data:   (31, 7)
Template:        (12000, 2)

BASIC DATA CHECKS

Training columns:
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']

Validation columns:
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']

December columns:
['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']

Template columns:
['load_id', 'predicted_rate']

TARGET
Target column: posted_rate

CREATING FINAL FEATURES
Development features: (48000, 45)
Validation features:  (12000, 44)
December features:    (31, 31)

ALIGNING FINAL MODEL FEATURES
Co